In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_transformer, make_column_selector
from sklearn.metrics import r2_score, log_loss, accuracy_score, roc_auc_score, log_loss

from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNet
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.ensemble import VotingClassifier, VotingRegressor

from sklearn.ensemble import BaggingClassifier

import warnings
warnings.filterwarnings("ignore")

In [4]:
kyph = pd.read_csv('D:/Machine_Learning/Cases/Kyphosis/Kyphosis.csv')

X = kyph.drop('Kyphosis',axis=1)
y=kyph['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=y)

### KNeighborsClassifier

In [6]:
knn = KNeighborsClassifier(n_neighbors=3)
bagg = BaggingClassifier( random_state=26, estimator=knn, n_estimators=15)
bagg.fit(X_train, y_train)
y_pred = bagg.predict(X_test)
accuracy_score(y_test, y_pred)

0.8

In [15]:
Ks = np.arange(1,11)
# nest = np.arange(1,20)
scores = []
for k in tqdm(Ks):
    knn = KNeighborsClassifier(n_neighbors=k)
    bagg = BaggingClassifier( random_state=26, estimator=knn, n_estimators=15)
    bagg.fit(X_train, y_train)
    y_pred = bagg.predict(X_test)
   
    scores.append([k, accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores, columns=['Neighbors', 'accuracy_score'])
df_scores.sort_values('accuracy_score', ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 23.64it/s]


,Neighbors,accuracy_score
1,2,0.80
2,3,0.80
6,7,0.80
3,4,0.80
4,5,0.80
5,6,0.80
8,9,0.80
7,8,0.80
9,10,0.80
0,1,0.72


In [14]:
Ks = np.arange(1,11)
# nest = np.arange(1,20)
scores = []
for k in tqdm(Ks):
    knn = KNeighborsClassifier(n_neighbors=k)
    bagg = BaggingClassifier( random_state=26, estimator=knn, n_estimators=15)
    bagg.fit(X_train, y_train)
    y_pred_prob = bagg.predict_proba(X_test)
    scores.append([k, log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['Neighbors', 'log_loss'])
df_scores.sort_values('log_loss', ascending=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 22.46it/s]


,Neighbors,log_loss
9,10,0.624843
8,9,0.651750
7,8,1.910555
6,7,1.934501
5,6,1.977256
4,5,2.021348
3,4,4.604602
2,3,4.608046
1,2,4.668663
0,1,5.994604


### LogisticRegression

In [17]:
Cs = np.linspace(0.001,5,20)
# nest = np.arange(1,20)
scores = []
for c in tqdm(Cs):
    lr = LogisticRegression(C=c)
    bagg = BaggingClassifier( random_state=26, estimator=lr, n_estimators=15)
    bagg.fit(X_train, y_train)
    y_pred_prob = bagg.predict_proba(X_test)
    scores.append([c, log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['C', 'log_loss'])
df_scores.sort_values('log_loss', ascending=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.37it/s]


,C,log_loss
1,0.264105,0.354583
2,0.527211,0.358009
3,0.790316,0.359767
4,1.053421,0.360808
5,1.316526,0.361487
6,1.579632,0.361961
7,1.842737,0.362308
8,2.105842,0.362574
9,2.368947,0.362783
10,2.632053,0.362951


### chech for alls

In [19]:
dtc = DecisionTreeClassifier(random_state=26)
knn = KNeighborsClassifier()
lr = LogisticRegression()
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()

bagg = BaggingClassifier( random_state=26, estimator=[('Tree', dtc),('KNN', knn), ('LR', lr), ('LDA', lda), ('NB', nb)], n_estimators=15)

estims = [dtc, knn, lr, lda, nb]
for e in estims:
    e.fit(X_train, y_train)
    y_pred_prob = e.predict_proba(X_test)
    print("Log_Loss of", e, "=", log_loss(y_test, y_pred_prob) )

Log_Loss of DecisionTreeClassifier(random_state=26) = 11.533969084517489
Log_Loss of KNeighborsClassifier() = 4.618638758809963
Log_Loss of LogisticRegression() = 0.3357708471894506
Log_Loss of LinearDiscriminantAnalysis() = 0.34281059610597553
Log_Loss of GaussianNB() = 0.39698492604441094


In [23]:
estims = [dtc, knn, lr, lda, nb]
scores = []

for e in tqdm(estims):
    for n in [10,15,50,100]:
        bagg = BaggingClassifier( random_state=26, estimator=e, n_estimators=n)
        bagg.fit(X_train, y_train)
        y_pred_prob = bagg.predict_proba(X_test)
        scores.append([e, n, log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['E','n','log_loss'])
df_scores.sort_values('log_loss', ascending=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:13<00:00,  2.69s/it]


,E,n,log_loss
10,LogisticRegression(),50,0.351872
14,LinearDiscriminantAnalysis(),50,0.354924
11,LogisticRegression(),100,0.355435
9,LogisticRegression(),15,0.360632
15,LinearDiscriminantAnalysis(),100,0.360662
13,LinearDiscriminantAnalysis(),15,0.363842
8,LogisticRegression(),10,0.363875
12,LinearDiscriminantAnalysis(),10,0.369988
18,GaussianNB(),50,0.411576
19,GaussianNB(),100,0.420313


# HR

In [24]:
hr = pd.read_csv('D:/Machine_Learning/Cases/HumanResource/HR_comma_sep.csv')

X = hr.drop('left',axis=1)
y=hr['left']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=y)

In [25]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
                            verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)
                            

In [26]:
dtc = DecisionTreeClassifier(random_state=26)
knn = KNeighborsClassifier()
lr = LogisticRegression()
lda = LinearDiscriminantAnalysis()
nb = GaussianNB()

# bagg = BaggingClassifier( random_state=26, estimator=[('Tree', dtc),('KNN', knn), ('LR', lr), ('LDA', lda), ('NB', nb)], n_estimators=15)

estims = [dtc, knn, lr, lda, nb]
scores = []

for e in tqdm(estims):
    for n in [10,15,50,100]:
        bagg = BaggingClassifier( random_state=26, estimator=e, n_estimators=n)
        bagg.fit(X_trn_ohe, y_train)
        y_pred_prob = bagg.predict_proba(X_tst_ohe)
        scores.append([e, n, log_loss(y_test, y_pred_prob)])

df_scores = pd.DataFrame(scores, columns=['E','n','log_loss'])
df_scores.sort_values('log_loss', ascending=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:09<00:00, 13.82s/it]


,E,n,log_loss
3,DecisionTreeClassifier(random_state=26),100,0.132272
2,DecisionTreeClassifier(random_state=26),50,0.173633
1,DecisionTreeClassifier(random_state=26),15,0.216398
0,DecisionTreeClassifier(random_state=26),10,0.261183
7,KNeighborsClassifier(),100,0.382559
6,KNeighborsClassifier(),50,0.383180
5,KNeighborsClassifier(),15,0.412145
12,LinearDiscriminantAnalysis(),10,0.430092
13,LinearDiscriminantAnalysis(),15,0.430226
11,LogisticRegression(),100,0.430226
